# Wan 2.1 I2V на Colab — видео «из фото» (бесплатный GPU T4, 16 ГБ)

Ноутбук по **одной фотографии** и **текстовому описанию движения** генерирует короткое видео (4–6 секунд), затем повышает качество до **720p / 1080p**.

**Что внутри:**
- Модель [Wan 2.1 I2V 14B](https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf) в кванте GGUF Q4_K_M (~11,3 ГБ) — единственный вариант, влезающий в 16 ГБ T4;
- Текстовый энкодер UMT5-XXL в кванте GGUF Q4_K_M (экономит ОЗУ ~12,7 ГБ);
- Движок [ComfyUI](https://github.com/Comfy-Org/ComfyUI) v0.36.0 + плагин [ComfyUI-GGUF](https://github.com/city96/ComfyUI-GGUF);
- Апскейл [Real-ESRGAN x4plus](https://github.com/xinntao/Real-ESRGAN) до 1080p.

**Лицензии (Apache-2.0):** Wan 2.1, GGUF-кванты, Real-ESRGAN; ComfyUI — GPL-3.0. Все модели берутся из публичных репозиториев — аккаунт и токен Hugging Face **не нужны**.

**Как запустить:** *Runtime → Change runtime type → Hardware accelerator: GPU T4* → затем *Runtime → Run all*. Первый запуск скачивает ~17 ГБ (10–40 мин), дальше всё занимает минуты.


## Как пользоваться — пошагово

1. **Включите GPU T4:** *Runtime → Change runtime type → Hardware accelerator: GPU T4 → Save*. Далее *Runtime → Run all* — ячейки 1–4 установят ComfyUI и скачают модели.
2. **Загрузите фото** в ячейке «Загрузка фото» (кнопка *Choose Files*). Лучше всего: широкое, чёткое, с хорошим светом фото без водяных знаков.
3. В ячейке **«Параметры видео»** опишите движение **по-английски** и при желании поменяйте длину (кадры), шаги и финальный размер.
4. Запустите ячейку **«Генерация»** — идёт диффузия + апскейл (~5–10 мин, виден лог).
5. Ячейка **«Сохранение»** покажет видео прямо в ноутбуке и предложит скачать MP4.

**Пример промпта (текст движения):**
```
A cute dog runs along the beach, the camera follows it, gentle waves,
soft warm light, cinematic shooting, 5.1 seconds.
```
> В конце промпта **обязательно** укажите длительность вида `5.1 seconds` — Wan опирается на неё.


## Лимиты и важные нюансы

| Параметр | Значение | Примечание |
|---|---|---|
| GPU / VRAM | T4 / 16 ГБ | Бесплатный Colab. При длительном простое сессия отключается, диск Colab очищается — модели придётся качать заново (см. шаг 11 про бэкап в Drive). |
| ОЗУ | ~12,7 ГБ | Поэтому модель в кванте GGUF Q4, а не в полном bf16. |
| Нативное разрешение | 832×480 | Wan 2.1 I2V 480P всегда генерирует такое; дальше апскейл. |
| Финальный размер | 1920×1080 или 1280×720 | Задаётся в параметрах (UPS_W / UPS_H). |
| Длина | 65–97 кадров | 65 ≈ 4,1 с, 81 ≈ 5,1 с, 97 ≈ 6,1 с при 16 fps. Длиннее нельзя — качество падает. |
| FPS | 16 | Стандарт Wan, не менять. |
| Шаги диффузии | 20 (по умолчанию) | 30–40 — лучше детали, но медленнее; 15 — черновик. |
| Время на видео | ~5–10 мин | Вечером (пиковая нагрузка) T4 работает медленнее. |
| Диск | ~17–18 ГБ | Модели + ComfyUI + зависимости. |
| Промпт | англ. / кит. | Кириллицу Wan понимает заметно хуже — переводите. |
| Артефакты | норма | Лица, руки и мелкие детали на движении — слабое место 14B на 16 ГБ. |

**Если Out of Memory:** сначала `LENGTH = 65`, затем `STEPS = 15` (ячейка «Параметры»). Если не помогло — *Runtime → Restart session* и повторите ячейки с нужного места (модели на диске останутся).


In [ ]:
# 1. Проверка окружения: GPU, ОЗУ, диск
import sys, os, time, json, uuid, random, socket, subprocess
import requests

print("Python:", sys.version.split()[0])
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
mem_gb = round(int(open("/proc/meminfo").read().split()[1]) / 1048576, 1)
print("ОЗУ:", mem_gb, "ГБ")
!df -h /content | tail -1
print("Ожидается NVIDIA T4 (16 ГБ) и ОЗУ ~12.7 ГБ.")


In [ ]:
# 2. Установка ComfyUI v0.36.0 и плагина ComfyUI-GGUF (идемпотентно)
COMFY_DIR = "/content/comfyui"
GGUF_DIR  = os.path.join(COMFY_DIR, "custom_nodes", "ComfyUI-GGUF")
COMFY_TAG = "v0.36.0"

os.makedirs(COMFY_DIR, exist_ok=True)
if not os.path.exists(os.path.join(COMFY_DIR, "main.py")):
    print("Клонирую ComfyUI", COMFY_TAG, "...")
    !git clone --depth 1 --branch {COMFY_TAG} https://github.com/Comfy-Org/ComfyUI.git {COMFY_DIR}
else:
    print("ComfyUI уже установлен — пропускаю клонирование.")

if not os.path.exists(os.path.join(GGUF_DIR, "nodes.py")):
    print("Клонирую плагин ComfyUI-GGUF ...")
    !git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git {GGUF_DIR}
else:
    print("Плагин GGUF уже установлен.")

# Зависимости движка и плагина (torch в Colab уже есть, pip его не заменяет)
!pip install -q -r {COMFY_DIR}/requirements.txt
!pip install -q -r {GGUF_DIR}/requirements.txt
print("Установка завершена.")


## (необязательно) Быстрый повторный запуск
Если в прошлый раз вы сохранили бэкап моделей на Google Drive (шаг 11), запустите сначала ячейку восстановления ниже — она распакует ~17 ГБ и шаг 4 пропустит скачивание. Если бэкапа нет, ячейка безвредна (просто ничего не сделает).


In [ ]:
# 3. (необязательно) Восстановление моделей из Google Drive
BACKUP = "/content/drive/MyDrive/comfy_backup/wan_models.tar"
if os.path.exists(BACKUP):
    print("Найден бэкап, распаковываю...")
    !tar -xf {BACKUP} -C /content/comfyui
    print("Модели восстановлены с Drive.")
else:
    print("Бэкапа нет — файлы будут скачаны на следующем шаге.")
    print("Подсказка: если Drive не смонтирован (папки /content/drive нет), запустите одну из ячеек монтирования (шаг 3 или 11).")


In [ ]:
# 4. Скачивание моделей (~17 ГБ). Повторный запуск докачивает недостающее.
MODELS = os.path.join(COMFY_DIR, "models")
for d in ["diffusion_models", "text_encoders", "vae", "clip_vision", "upscale_models"]:
    os.makedirs(os.path.join(MODELS, d), exist_ok=True)

JOBS = [
    # (url, папка в models/, имя файла, минимальный размер в ГБ)
    ("https://huggingface.co/city96/Wan2.1-I2V-14B-480P-gguf/resolve/main/wan2.1-i2v-14b-480p-Q4_K_M.gguf",
     "diffusion_models", "wan2.1-i2v-14b-480p-Q4_K_M.gguf", 10.6),
    ("https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q4_K_M.gguf",
     "text_encoders", "umt5-xxl-encoder-Q4_K_M.gguf", 3.4),
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
     "vae", "wan_2.1_vae.safetensors", 0.25),
    ("https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
     "clip_vision", "clip_vision_h.safetensors", 1.2),
    ("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
     "upscale_models", "RealESRGAN_x4plus.pth", 0.05),
]

def download(url, folder, name, min_gb):
    dest = os.path.join(MODELS, folder, name)
    if os.path.exists(dest) and os.path.getsize(dest) > min_gb * 0.9e9:
        print("[уже есть]", name)
        return True
    print("Качаю:", name)
    t0 = time.time()
    !wget -c --tries=4 --timeout=60 --no-verbose -O {dest} {url}
    ok = os.path.exists(dest) and os.path.getsize(dest) > min_gb * 0.9e9
    size = round(os.path.getsize(dest) / 1e9, 2) if os.path.exists(dest) else 0
    print(("OK   " if ok else "СБОЙ "), name, size, "ГБ", "[" + str(int(time.time() - t0)) + " c]")
    return ok

ok_all = True
for u, f, n, m in JOBS:
    ok_all = download(u, f, n, m) and ok_all

if not ok_all:
    raise SystemExit("Один из файлов не докачался. Проверьте интернет и перезапустите ячейку.")
print("Все модели на месте:")
!du -sh {MODELS}


In [ ]:
# 5. Запуск сервера ComfyUI в фоне (порт 8188)
COMFY_PORT = 8188
BASE = f"http://127.0.0.1:{COMFY_PORT}"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def port_open(port):
    s = socket.socket(); s.settimeout(1)
    try:
        s.connect(("127.0.0.1", port)); return True
    except Exception:
        return False
    finally:
        s.close()

if port_open(COMFY_PORT):
    print("Сервер уже запущен — пропускаю.")
else:
    logf = open("/content/comfyui_server.log", "w")
    subprocess.Popen(
        [sys.executable, "main.py", "--port", str(COMFY_PORT), "--disable-auto-launch"],
        cwd=COMFY_DIR, stdout=logf, stderr=subprocess.STDOUT)
    started = False
    for i in range(300):   # до 10 минут (первый запуск — компиляция)
        time.sleep(2)
        try:
            if requests.get(BASE + "/system_stats", timeout=3).status_code == 200:
                started = True
                break
        except Exception:
            pass
    if started:
        print("Сервер готов за", (i + 1) * 2, "сек")
    else:
        print("Сервер не поднялся. Последние строки лога:")
        print(open("/content/comfyui_server.log").read()[-4000:])
        raise SystemExit("Смотрите лог выше")


In [ ]:
# 6. Проверка узлов и автовыбор имён моделей
object_info = requests.get(BASE + "/object_info").json()

required = ["UnetLoaderGGUF", "CLIPLoaderGGUF", "CLIPVisionLoader", "CLIPVisionEncode",
            "CLIPTextEncode", "VAELoader", "VAEDecode", "LoadImage", "WanImageToVideo",
            "ModelSamplingSD3", "KSampler", "ImageUpscaleWithModel", "ImageScale",
            "CreateVideo", "SaveVideo"]
missing = [n for n in required if n not in object_info]
if missing:
    print("Не найдены узлы:", missing)
    if "UnetLoaderGGUF" in missing or "CLIPLoaderGGUF" in missing:
        print("Плагин GGUF не установился. Решение: Runtime → Restart session, затем шаги 2, 5, 6.")
    raise SystemExit(1)
print("Все ключевые узлы на месте.")

def input_names(cls):
    inp = object_info[cls]["input"]
    return set(inp.get("required", {})) | set(inp.get("optional", {}))

def combo_options(spec):
    if not isinstance(spec, (list, tuple)) or len(spec) < 2:
        return None
    c, props = spec[0], spec[1]
    if isinstance(c, list) and c:
        return [s for s in c if isinstance(s, str)]
    if isinstance(props, dict) and isinstance(props.get("options"), list):
        return [s for s in props["options"] if isinstance(s, str)]
    if isinstance(c, dict) and isinstance(c.get("options"), list):
        return [s for s in c["options"] if isinstance(s, str)]
    if isinstance(c, str):
        return [c]
    return None

def pick(cls, field, filename):
    inp = object_info[cls]["input"]
    for sec in ("required", "optional"):
        if field in inp.get(sec, {}):
            opts = combo_options(inp[sec][field])
            if opts:
                return opts[0]
    return filename

UNET_NAME = pick("UnetLoaderGGUF", "unet_name", "wan2.1-i2v-14b-480p-Q4_K_M.gguf")
CLIP_NAME = pick("CLIPLoaderGGUF", "clip_name", "umt5-xxl-encoder-Q4_K_M.gguf")
VAE_NAME  = pick("VAELoader", "vae_name", "wan_2.1_vae.safetensors")
CV_NAME   = pick("CLIPVisionLoader", "clip_name", "clip_vision_h.safetensors")
UP_NAME   = pick("UpscaleModelLoader", "model_name", "RealESRGAN_x4plus.pth")

print("UNet (GGUF):      ", UNET_NAME)
print("Текст. энкодер:   ", CLIP_NAME)
print("VAE:              ", VAE_NAME)
print("CLIP Vision:      ", CV_NAME)
print("Апскейлер:        ", UP_NAME)
NODE_IN = {c: input_names(c) for c in required}
print("Входы SaveVideo:", sorted(input_names("SaveVideo")))
print("Входы CreateVideo:", sorted(input_names("CreateVideo")))


In [ ]:
# 7. Загрузка фото (одно, JPG/PNG)
from google.colab import files
print("Нажмите кнопку и выберите ОДНО фото (JPG/PNG).")
UP = files.upload()
if not UP:
    raise SystemExit("Файл не выбран")
PHOTO = list(UP.keys())[0]
os.makedirs(os.path.join(COMFY_DIR, "input"), exist_ok=True)
import shutil
shutil.copy(PHOTO, os.path.join(COMFY_DIR, "input", PHOTO))
print("Фото загружено:", PHOTO)


In [ ]:
# 8. Параметры видео: промпт движения, длина, шаги, финальный размер
# Промпт — на английском, в конце ОБЯЗАТЕЛЬНО длительность вида "5.1 seconds".
POSITIVE = ("A cute dog runs along the beach, the camera follows it, "
            "gentle waves crashing on the shore, soft warm light, "
            "cinematic shooting, 5.1 seconds.")

# Отрицательный промпт (стандартный шаблон Wan) — обычно менять не нужно
NEGATIVE = ("模糊景物, 模糊画面, 低质量, 扭曲, 变形, 重复, 过度曝光, 视频水印, 文字叠加, "
            "静态画面, 严重压缩痕迹, 丑陋, 不完整, 多余的手指, 手部扭曲, 面部扭曲, 残缺, 连接的手指")

# ── Технические параметры ──
WAN_RES = (832, 480)   # нативное разрешение модели 480P — не менять
LENGTH  = 81           # кадров: 65≈4.1с | 81≈5.1с | 97≈6.1с
STEPS   = 20           # 20 — баланс, 30–40 — качество, 15 — черновик
CFG     = 6.0          # сила следования промпту (обычно 5–8)
SEED    = random.randint(0, 2 ** 31)
FPS     = 16           # стандарт Wan
UPS_W, UPS_H = 1920, 1080   # 1920x1080 или 1280x720

print("Движение:", POSITIVE)
print(f"Кадр: {WAN_RES[0]}x{WAN_RES[1]} | кадров: {LENGTH} ({round(LENGTH / FPS, 2)} с) | шагов: {STEPS} | CFG: {CFG}")
print(f"Seed: {SEED} | финальный размер: {UPS_W}x{UPS_H}")


In [ ]:
# 9. Генерация: сборка графа (адаптивно под версию ComfyUI), диффузия + апскейл + рендер
def link(nid, idx):
    return [str(nid), idx]

graph = {
    "4":  {"class_type": "CLIPLoaderGGUF",      "inputs": {"clip_name": CLIP_NAME, "type": "wan"}},
    "5":  {"class_type": "UnetLoaderGGUF",      "inputs": {"unet_name": UNET_NAME}},
    "6":  {"class_type": "CLIPVisionLoader",    "inputs": {"clip_name": CV_NAME}},
    "7":  {"class_type": "CLIPVisionEncode",    "inputs": {"clip_vision": link(6, 0), "image": link(10, 0), "crop": "center"}},
    "8":  {"class_type": "CLIPTextEncode",      "inputs": {"text": POSITIVE, "clip": link(4, 0)}},
    "9":  {"class_type": "CLIPTextEncode",      "inputs": {"text": NEGATIVE, "clip": link(4, 0)}},
    "10": {"class_type": "LoadImage",           "inputs": {"image": PHOTO}},
    "11": {"class_type": "VAELoader",           "inputs": {"vae_name": VAE_NAME}},
    "12": {"class_type": "WanImageToVideo",     "inputs": {
        "positive": link(8, 0), "negative": link(9, 0), "vae": link(11, 0),
        "width": WAN_RES[0], "height": WAN_RES[1], "length": LENGTH, "batch_size": 1,
        "clip_vision_output": link(7, 0), "start_image": link(10, 0)}},
    "13": {"class_type": "ModelSamplingSD3",    "inputs": {"model": link(5, 0), "shift": 8.0}},
    "14": {"class_type": "KSampler",            "inputs": {
        "model": link(13, 0), "seed": SEED, "steps": STEPS, "cfg": CFG,
        "sampler_name": "uni_pc", "scheduler": "simple",
        "positive": link(12, 0), "negative": link(12, 1), "latent_image": link(12, 2), "denoise": 1.0}},
    "15": {"class_type": "VAEDecode",           "inputs": {"samples": link(14, 0), "vae": link(11, 0)}},
    "16": {"class_type": "UpscaleModelLoader",  "inputs": {"model_name": UP_NAME}},
    "17": {"class_type": "ImageUpscaleWithModel", "inputs": {"upscale_model": link(16, 0), "image": link(15, 0)}},
    "18": {"class_type": "ImageScale",          "inputs": {"image": link(17, 0), "upscale_method": "lanczos", "width": UPS_W, "height": UPS_H, "crop": "disabled"}},
}

SAVE_IN = input_names("SaveVideo")
if "video" in SAVE_IN:
    # Новый ComfyUI: images -> CreateVideo -> SaveVideo(video)
    graph["19"] = {"class_type": "CreateVideo", "inputs": {"images": link(18, 0), "fps": FPS}}
    save_inputs = {"video": link(19, 0), "filename_prefix": "wan_video"}
    if "fps" in input_names("SaveVideo"):
        save_inputs["fps"] = FPS
    if "format" in SAVE_IN:
        is_req = "format" in object_info["SaveVideo"]["input"].get("required", {})
        save_inputs["format"] = "video/h264-mp4" if is_req else "auto"
    graph["20"] = {"class_type": "SaveVideo", "inputs": save_inputs}
    OUT_NODE = "20"
else:
    # Старый ComfyUI: SaveVideo принимает images напрямую
    save_inputs = {"images": link(18, 0), "filename_prefix": "wan_video", "fps": FPS}
    if "format" in SAVE_IN:
        save_inputs["format"] = "video/h264-mp4"
    graph["19"] = {"class_type": "SaveVideo", "inputs": save_inputs}
    OUT_NODE = "19"

# Страховка от изменения версий: убираем входы, которых нет в объекте
for nid in list(graph):
    cls = graph[nid]["class_type"]
    inp = object_info[cls]["input"]
    keys = set(inp.get("required", {})) | set(inp.get("optional", {}))
    unknown = [k for k in graph[nid]["inputs"] if k not in keys]
    if unknown:
        print(f"Узел {cls}: пропущены неизвестные входы {unknown}")
    graph[nid]["inputs"] = {k: v for k, v in graph[nid]["inputs"].items() if k in keys}

# Проверка, что все обязательные входы заполнены
for nid in list(graph):
    cls = graph[nid]["class_type"]
    for need in object_info[cls]["input"].get("required", {}):
        if need not in graph[nid]["inputs"]:
            raise SystemExit(f"Узел {cls}: не хватает обязательного входа '{need}'")

# Отправка задачи
r = requests.post(BASE + "/prompt", json={"prompt": graph}, timeout=30)
if r.status_code != 200:
    print("Ошибка при старте генерации:")
    print(r.text[:4000])
    raise SystemExit(1)
pid = r.json()["prompt_id"]
print("Задача запущена. Диффузия + апскейл, ~5–10 мин. Ожидание...")
t0 = time.time()
while True:
    time.sleep(5)
    h = requests.get(f"{BASE}/history/{pid}").json()
    if pid in h:
        break
    el = int(time.time() - t0)
    print(f"... идёт генерация ({el // 60} мин {el % 60} с)")
hitem = h[pid]
if hitem.get("status", {}).get("status_str") == "error":
    print("Ошибка выполнения:")
    for m in hitem["status"].get("messages", []):
        print(m)
    raise SystemExit(1)
print(f"Готово за {int(time.time() - t0)} с.")

# Ищем файл результата
item = None
for oid, ov in hitem.get("outputs", {}).items():
    for k in ("gifs", "videos"):
        if k in ov and isinstance(ov[k], list) and ov[k]:
            item = ov[k][0]
            break
    if item:
        break
if item is None:
    print("Результат не найден, полный ответ:")
    print(json.dumps(hitem, indent=1, ensure_ascii=False)[:4000])
    raise SystemExit(1)
ITEM = item
print("Файл сохранён сервером:", ITEM["filename"])


In [ ]:
# 10. Скачивание, просмотр и сохранение результата
fname = ITEM["filename"]
subf  = ITEM.get("subfolder") or ""
ftype = ITEM.get("type") or "output"
params = {"filename": fname, "subfolder": subf, "type": ftype}
fm = ITEM.get("format")
if isinstance(fm, str) and fm.startswith("video/"):
    params["format"] = fm
r = requests.get(BASE + "/view", params=params, timeout=300)
r.raise_for_status()
dest = "/content/wan_video_" + time.strftime("%H%M%S") + ".mp4"
with open(dest, "wb") as f:
    f.write(r.content)
print("Сохранено:", dest, f"({round(os.path.getsize(dest) / 1e6, 1)} МБ)")

# Встроенный просмотр (если файл не слишком большой)
from IPython.display import display, Video
if os.path.getsize(dest) < 30e6:
    display(Video(dest, embed=True))
else:
    print("Файл большой — превью пропущено, видео лежит в", dest)

# Кнопка скачивания в браузер
from google.colab import files
files.download(dest)
print("Готово! Если диалог не открылся — файл в /content/.")


In [ ]:
# 11. (необязательно) Сохранить модели на Google Drive (одноразово, ~17 ГБ)
# В следующий раз: смонтируйте Drive и запустите шаг 3 — скачивание 17 ГБ пропадёт.
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/comfy_backup", exist_ok=True)
backup = "/content/drive/MyDrive/comfy_backup/wan_models.tar"
if os.path.exists(backup):
    print("Бэкап уже есть — не перезаписываю.")
else:
    print("Упаковываю модели, минут 5–10...")
    !tar -cf {backup} -C /content/comfyui models
    !ls -lh {backup}
print("Готово.")


In [ ]:
# 12. Туннель cloudflared для удалённого доступа (нужен локальному wanbox)
import os, subprocess, time, urllib.request, re

CF = "/content/cloudflared"
LOG = "/content/wanbox_tunnel.log"
URLF = "/content/wanbox_tunnel.txt"

def tunnel_url():
    try:
        txt = open(URLF).read().strip()
        if txt.startswith("https://"):
            return txt
    except Exception:
        pass
    try:
        data = open(LOG).read()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", data)
        if m:
            open(URLF, "w").write(m.group(0))
            return m.group(0)
    except Exception:
        pass
    return None

if tunnel_url():
    print("Туннель уже запущен:", tunnel_url())
else:
    if not os.path.exists(CF):
        print("Скачиваю cloudflared...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CF)
        os.chmod(CF, 0o755)
    logf = open(LOG, "w")
    subprocess.Popen([CF, "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                     stdout=logf, stderr=subprocess.STDOUT)
    url = None
    for i in range(60):
        time.sleep(2)
        url = tunnel_url()
        if url:
            break
    if url:
        print("Туннель:", url)
        print("Вставьте этот URL в wanbox (поле ComfyUI colab-профиля). Работает, пока сессия Colab жива.")
    else:
        print("Туннель не поднялся. Последние строки лога:")
        print(open(LOG).read()[-2000:])


In [ ]:
# 12b. Показать текущий URL туннеля
try:
    print(open("/content/wanbox_tunnel.txt").read().strip())
except Exception:
    print("Туннель не запущен — выполните ячейку 12.")


## Частые вопросы и решение проблем

**«Out of Memory» — не хватает VRAM.** Сначала `LENGTH = 65`, затем `STEPS = 15` в ячейке «Параметры». Не помогло — *Runtime → Restart session* и запустите ячейки заново (модели на диске останутся).

**Узел `UnetLoaderGGUF` не найден (MISSING).** Плагин GGUF не установился. *Runtime → Restart session* → перезапустите шаги 2, 5, 6.

**Сервер не поднялся / узлы не найдены.** Смотрите последние строки `/content/comfyui_server.log` (печатаются автоматически в шаге 5). Частая причина — нехватка памяти при инициализации; помогает перезапуск сессии.

**Можно ли 720p?** Да: в «Параметрах» `UPS_W, UPS_H = 1280, 720` — быстрее и надёжнее, чем 1080p.

**Промпт по-русски не работает.** Wan обучен на англ./кит. — переводите.

**Лицо/руки «плывут», видео мигает.** Это норма для 16 ГБ. Помогает `STEPS` 30–40, малое движение в тексте и длина 65 кадров.

**Модели пропали после перезапуска?** Бесплатный Colab очищает диск. Используйте шаг 11 (бэкап в Drive) и восстановление на шаге 3.

## Как это устроено внутри
Фото → CLIP Vision (сцена) + ваш текст → latent → 20 шагов диффузии Wan 2.1 14B (в кванте Q4_K_M) → 81 кадр 832×480 → RealESRGAN ×4 → ресайз в 1920×1080 → H.264 MP4 (16 fps). Квант Q4 экономит ~4 ГБ VRAM — именно это делает запуск на T4 возможным; на финальном «фотореалистичном» стиле он почти не сказывается.
